# GenAI Adoption vs. GDP Analysis

**Question:** Is real-world GenAI adoption really as widespread as advertising and media coverage suggest, and how does it relate to GDP per capita?

**Data source:** [Our World in Data — Generative AI vs. GDP per Capita](https://ourworldindata.org/) (CC BY)

This notebook was built collaboratively with Claude as part of Anthropic's AI Fluency course. See `progress-notes.md` and `project-plan.md` in this repo for the full breakdown of how work was divided between human judgment and AI assistance. A lot of the analysis was conducted as a back and forth conversation with Claude, speaking about what challenged my assumptions and discussing outliers as well as surprising features of this analysis.


In [4]:
import pandas as pd

df = pd.read_csv('../data/cleaned/genai_adoption_clean_full.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()


,country,code,date,adoption_pct,gdp_per_capita,region
0,Afghanistan,AFG,2025-06-30,5.1,1983.8126,Asia
1,Afghanistan,AFG,2025-12-31,5.6,1983.8126,Asia
2,Afghanistan,AFG,2026-03-31,6.1,1983.8126,Asia
3,Albania,ALB,2025-06-30,15.8,21641.0740,Europe
4,Albania,ALB,2025-12-31,16.5,21641.0740,Europe


## Question 1: Does GDP per capita correlate with GenAI adoption?

Using the March 2026 snapshot to avoid counting each country three times.


In [5]:
snapshot = df[df['date'] == '2026-03-31'].dropna(subset=['gdp_per_capita'])
corr = snapshot['gdp_per_capita'].corr(snapshot['adoption_pct'])
print(f"Correlation between GDP per capita and adoption rate: {corr:.3f}")
print(f"Countries in this snapshot: {len(snapshot)}")


Correlation between GDP per capita and adoption rate: 0.846
Countries in this snapshot: 141


**Finding:** Correlation is 0.846 — strong, but not perfect.

**Interpretation:** My original assumption was close to a one-to-one relationship between GDP and adoption. This mostly holds — likely driven by infrastructure, knowledge-work jobs, and the availability of English-language training content — but it isn't a perfect 1:1 relationship, which leaves room for the outliers found in Question 3 below.


## Question 2: Which countries saw adoption decrease between June 2025 and March 2026?


In [6]:
early = df[df['date'] == '2025-06-30'][['country', 'adoption_pct']].rename(columns={'adoption_pct': 'early_pct'})
late = df[df['date'] == '2026-03-31'][['country', 'adoption_pct']].rename(columns={'adoption_pct': 'late_pct'})
change = early.merge(late, on='country')
change['change'] = change['late_pct'] - change['early_pct']

declined = change[change['change'] < 0].sort_values('change')
print(f"Countries with declining adoption: {len(declined)}")
print("\nSmallest growth (closest to declining):")
print(change.sort_values('change').head(5))


Countries with declining adoption: 0

Smallest growth (closest to declining):
              country  early_pct  late_pct  change
137           Ukraine        9.1       9.4     0.3
100  Papua New Guinea        7.2       7.7     0.5
127             Syria        6.7       7.5     0.8
46             Gambia       10.6      11.4     0.8
69              Kenya        7.8       8.7     0.9


**Finding:** Zero countries declined. Every one of 147 countries grew between June 2025 and March 2026.

**Interpretation:** This is a genuinely notable pattern, not a coincidence. With this many countries, some random decline would be expected by chance if growth weren't broadly consistent — a clean zero suggests adoption growth is real and widespread, not just an average trend masking mixed results underneath.


## Question 3: Which countries have adoption higher or lower than their GDP rank would predict?

Comparing each country's GDP rank to its adoption rank (March 2026) to find the largest gaps.


In [7]:
snapshot = df[df['date'] == '2026-03-31'].dropna(subset=['gdp_per_capita']).copy()
snapshot['gdp_rank'] = snapshot['gdp_per_capita'].rank(ascending=False)
snapshot['adoption_rank'] = snapshot['adoption_pct'].rank(ascending=False)
snapshot['rank_gap'] = snapshot['gdp_rank'] - snapshot['adoption_rank']

print("Over-performers (adoption higher than GDP rank predicts):")
print(snapshot.sort_values('rank_gap', ascending=False).head(5)[['country','gdp_rank','adoption_rank','rank_gap']])

print("\nUnder-performers (adoption lower than GDP rank predicts):")
print(snapshot.sort_values('rank_gap', ascending=True).head(5)[['country','gdp_rank','adoption_rank','rank_gap']])


Over-performers (adoption higher than GDP rank predicts):
        country  gdp_rank  adoption_rank  rank_gap
203      Jordan      91.0           26.5      64.5
221     Lebanon      85.0           32.0      53.0
263  Mozambique     138.0           93.5      44.5
239      Malawi     136.0           93.5      42.5
434     Vietnam      75.0           33.5      41.5

Under-performers (adoption lower than GDP rank predicts):
          country  gdp_rank  adoption_rank  rank_gap
164        Guyana       8.0           96.5     -88.5
326        Russia      34.0          113.5     -79.5
407  Turkmenistan      63.0          139.0     -76.0
17        Armenia      59.0          134.0     -75.0
32        Belarus      47.0          112.0     -65.0


**Finding:** Jordan, Lebanon, and Vietnam adopt well beyond what their GDP rank predicts. Guyana and Russia adopt well below what their GDP rank predicts.

**Interpretation:** These are genuine outliers from the general GDP-adoption trend — specifically "relationship outliers" (they break the pattern most countries follow) rather than simple extreme-value outliers. This finding directly challenged my original assumption of a near one-to-one GDP-adoption relationship.

**Caveat:** With only one data snapshot and just 5 of 141 countries showing large gaps, this is flagged as worth investigating further rather than explained with confidence. The data shows *that* these countries deviate, not *why*.


## Question 4: Is high adoption broad-based, or concentrated in a few standout countries?

Comparing the global median adoption rate to the average of the top 10 countries (March 2026).


In [8]:
snapshot = df[df['date'] == '2026-03-31']
median_val = snapshot['adoption_pct'].median()
top10 = snapshot.sort_values('adoption_pct', ascending=False).head(10)
top10_avg = top10['adoption_pct'].mean()

print(f"Median adoption (March 2026): {median_val:.1f}%")
print(f"Top 10 countries average: {top10_avg:.1f}%")
print(f"Gap: {top10_avg - median_val:.1f} percentage points")
print("\nTop 10 countries:")
print(top10[['country', 'adoption_pct']])


Median adoption (March 2026): 14.8%
Top 10 countries average: 49.2%
Gap: 34.4 percentage points

Top 10 countries:
                  country  adoption_pct
416  United Arab Emirates          70.1
344             Singapore          63.4
290                Norway          48.6
188               Ireland          48.4
131                France          47.8
365                 Spain          44.2
278           New Zealand          43.0
419        United Kingdom          42.2
275           Netherlands          42.1
320                 Qatar          41.8


**Finding:** Median adoption is 14.8%, but the top 10 countries average 49.2% — a 34.4 percentage point gap.

**Interpretation:** This is the strongest, most direct answer to the question that motivated this project: is AI adoption really as high as advertising implies? The answer is nuanced: the hype isn't false, but it describes a small group of leading countries (UAE, Singapore, Norway, and similar) and gets generalized to the whole world — which is misleading for the typical country, where adoption remains far lower.


## Question 5: Do regions cluster in adoption independent of GDP?

Comparing average adoption and average GDP per capita by world region (March 2026).


In [9]:
snapshot = df[df['date'] == '2026-03-31'].dropna(subset=['gdp_per_capita', 'region'])
region_stats = snapshot.groupby('region').agg(
    avg_adoption=('adoption_pct', 'mean'),
    avg_gdp=('gdp_per_capita', 'mean'),
    n_countries=('country', 'count')
).round(1)
region_stats['adoption_rank'] = region_stats['avg_adoption'].rank(ascending=False)
region_stats['gdp_rank'] = region_stats['avg_gdp'].rank(ascending=False)
region_stats = region_stats.sort_values('adoption_rank')

print(region_stats)


               avg_adoption  avg_gdp  n_countries  adoption_rank  gdp_rank
region                                                                    
Oceania                30.1  37969.6            3            1.0       2.0
Europe                 29.6  49749.2           32            2.0       1.0
North America          21.5  24545.1           12            3.0       5.0
Asia                   19.6  26774.5           39            4.0       3.0
South America          17.7  24912.9           11            5.0       4.0
Africa                 10.8   5761.9           44            6.0       6.0


**Finding:** Oceania and Europe lead in both average adoption and average GDP; Africa is lowest on both.

**Interpretation:** Expected, given the strength of the Question 1 correlation — this is largely the same wealth-adoption pattern showing up at the regional level. One nuance worth noting: North America and Asia don't rank identically for GDP vs. adoption, showing that region alone doesn't perfectly track wealth either.


## Summary

- GenAI adoption is strongly tied to GDP (0.846 correlation), but not perfectly — a handful of countries adopt well above or below what wealth alone predicts.
- Adoption grew in every single country studied between June 2025 and March 2026 — a genuinely consistent global trend.
- The headline "AI is everywhere" narrative reflects a small group of leading countries, not the typical country: the median nation sits at 14.8% adoption, well below the ~49% average of the top 10.
